In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "models").exists():
    PROJECT_ROOT = Path.cwd()

MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

WINDOW_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "CWRU"
    / "windows"
)

QUANT_DIR = RESULTS_DIR / "quantized"

print("Project root:", PROJECT_ROOT)
print("Window directory:", WINDOW_DIR)
print("Quantized directory:", QUANT_DIR)

Project root: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection
Window directory: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows
Quantized directory: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized


In [2]:
FP32_MODEL_PATH = MODEL_DIR / "vaac_tiny_best.keras"

fp32_model = tf.keras.models.load_model(
    FP32_MODEL_PATH
)

print("FP32 model loaded")
print("Parameters:", fp32_model.count_params())
print("Input shape:", fp32_model.input_shape)
print("Output shape:", fp32_model.output_shape)

FP32 model loaded
Parameters: 4244
Input shape: (None, 12000, 1)
Output shape: (None, 4)


In [3]:
INT8_MODEL_PATH = (
    QUANT_DIR / "vaac_tiny_int8.tflite"
)

print("INT8 model exists:",
      INT8_MODEL_PATH.exists())

print("INT8 model:",
      INT8_MODEL_PATH)

INT8 model exists: True
INT8 model: c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized\vaac_tiny_int8.tflite


In [4]:
interpreter = tf.lite.Interpreter(
    model_path=str(INT8_MODEL_PATH)
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("INT8 interpreter loaded successfully.")

print("\nInput dtype:",
      input_details[0]["dtype"])

print("Output dtype:",
      output_details[0]["dtype"])

INT8 interpreter loaded successfully.

Input dtype: <class 'numpy.int8'>
Output dtype: <class 'numpy.int8'>


c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\.venv\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [5]:
TEST_METADATA = WINDOW_DIR / "test_metadata.csv"

test_df = pd.read_csv(
    TEST_METADATA
)

print("Test metadata shape:",
      test_df.shape)

display(test_df.head())

Test metadata shape: (136, 8)


,recording_id,source_file,signal_id,class,window_id,start_sample,end_sample,label
0,B007_3_X121,B007_3.mat,X121,Ball,0,0,12000,1
1,B007_3_X121,B007_3.mat,X121,Ball,1,6000,18000,1
2,B007_3_X121,B007_3.mat,X121,Ball,2,12000,24000,1
3,B007_3_X121,B007_3.mat,X121,Ball,3,18000,30000,1
4,B007_3_X121,B007_3.mat,X121,Ball,4,24000,36000,1


In [6]:
CWRU_PROCESSED = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "CWRU"
)

all_test_npy = []

for path in CWRU_PROCESSED.rglob("*.npy"):
    
    path_string = str(path).lower()
    
    if "test" in path_string:
        all_test_npy.append(path)

print("Test .npy files found:",
      len(all_test_npy))

for path in all_test_npy[:10]:
    print(path)

Test .npy files found: 212
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0000.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0001.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0002.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0003.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0004.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0005.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows\test\B007_0_X118_window_0006.npy
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\proce

In [7]:
test_file_index = {}

for path in all_test_npy:
    test_file_index[path.name] = path

print("Indexed test files:",
      len(test_file_index))

Indexed test files: 212


In [9]:
def get_test_window_path(row):
    
    filename = (
        f"{row['recording_id']}"
        f"_window_{int(row['window_id']):04d}.npy"
    )
    
    if filename not in test_file_index:
        raise FileNotFoundError(
            f"Test window not found:\n{filename}"
        )
    
    return test_file_index[filename]
example_path = get_test_window_path(
    test_df.iloc[0]
)

print(example_path)
print("Exists:", example_path.exists())

c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\data\processed\CWRU\windows_corrected\test\B007_3_X121_window_0000.npy
Exists: True


In [10]:
missing_files = []

for _, row in test_df.iterrows():
    
    filename = (
        f"{row['recording_id']}"
        f"_window_{int(row['window_id']):04d}.npy"
    )
    
    if filename not in test_file_index:
        missing_files.append(filename)

print("Test windows:", len(test_df))
print("Missing files:", len(missing_files))

if missing_files:
    print("\nMissing files:")
    for filename in missing_files[:20]:
        print(filename)
else:
    print("\nAll test window files found.")

Test windows: 136
Missing files: 0

All test window files found.


In [11]:
CLASS_NAMES = [
    "Healthy",
    "Ball",
    "Inner Race",
    "Outer Race"
]

CLASS_TO_LABEL = {
    "Healthy": 0,
    "Ball": 1,
    "Inner Race": 2,
    "Outer Race": 3
}

LABEL_TO_CLASS = {
    value: key
    for key, value in CLASS_TO_LABEL.items()
}

print(CLASS_TO_LABEL)

{'Healthy': 0, 'Ball': 1, 'Inner Race': 2, 'Outer Race': 3}


In [12]:
y_test = test_df["class"].map(
    CLASS_TO_LABEL
)

print("True labels:")
print(y_test.value_counts().sort_index())

print("\nUnique labels:",
      sorted(y_test.unique()))

True labels:
class
0    79
1    19
2    19
3    19
Name: count, dtype: int64

Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [13]:
def predict_fp32(signal):
    
    signal = np.asarray(
        signal,
        dtype=np.float32
    )
    
    signal = signal.reshape(
        1, 12000, 1
    )
    
    probabilities = fp32_model.predict(
        signal,
        verbose=0
    )[0]
    
    predicted_label = int(
        np.argmax(probabilities)
    )
    
    confidence = float(
        np.max(probabilities)
    )
    
    return (
        predicted_label,
        confidence,
        probabilities
    )

In [14]:
input_scale, input_zero_point = (
    input_details[0]["quantization"]
)

output_scale, output_zero_point = (
    output_details[0]["quantization"]
)

print("Input scale:", input_scale)
print("Input zero point:", input_zero_point)

print("Output scale:", output_scale)
print("Output zero point:", output_zero_point)

Input scale: 0.027604417875409126
Input zero point: -5
Output scale: 0.00390625
Output zero point: -128


In [15]:
def predict_int8(signal):
    
    signal = np.asarray(
        signal,
        dtype=np.float32
    )
    
    signal = signal.reshape(
        1, 12000, 1
    )
    
    # Quantize input
    quantized_signal = np.round(
        signal / input_scale
    ) + input_zero_point
    
    quantized_signal = np.clip(
        quantized_signal,
        -128,
        127
    ).astype(np.int8)
    
    # Set input tensor
    interpreter.set_tensor(
        input_details[0]["index"],
        quantized_signal
    )
    
    # Run inference
    interpreter.invoke()
    
    # Get quantized output
    quantized_output = interpreter.get_tensor(
        output_details[0]["index"]
    )[0]
    
    # Dequantize output
    output = (
        output_scale *
        (
            quantized_output.astype(np.float32)
            - output_zero_point
        )
    )
    
    # Normalize for probability comparison
    probabilities = tf.nn.softmax(
        output
    ).numpy()
    
    predicted_label = int(
        np.argmax(probabilities)
    )
    
    confidence = float(
        np.max(probabilities)
    )
    
    return (
        predicted_label,
        confidence,
        probabilities
    )

In [16]:
test_row = test_df.iloc[0]

test_path = get_test_window_path(
    test_row
)

signal = np.load(
    test_path
)

print("Signal shape:", signal.shape)
print("Signal dtype:", signal.dtype)

Signal shape: (12000,)
Signal dtype: float64


In [17]:
fp32_pred, fp32_conf, fp32_prob = (
    predict_fp32(signal)
)

int8_pred, int8_conf, int8_prob = (
    predict_int8(signal)
)

print("True class:",
      test_row["class"])

print(
    "\nFP32 prediction:",
    LABEL_TO_CLASS[fp32_pred]
)

print(
    "FP32 confidence:",
    fp32_conf
)

print(
    "\nINT8 prediction:",
    LABEL_TO_CLASS[int8_pred]
)

print(
    "INT8 confidence:",
    int8_conf
)

True class: Ball

FP32 prediction: Outer Race
FP32 confidence: 0.9915315508842468

INT8 prediction: Outer Race
INT8 confidence: 0.47276797890663147


In [18]:
results = []

for index, row in test_df.iterrows():
    
    path = get_test_window_path(row)
    
    signal = np.load(path)
    
    true_label = CLASS_TO_LABEL[
        row["class"]
    ]
    
    # FP32
    fp32_pred, fp32_conf, fp32_prob = (
        predict_fp32(signal)
    )
    
    # INT8
    int8_pred, int8_conf, int8_prob = (
        predict_int8(signal)
    )
    
    results.append({
        "recording_id":
            row["recording_id"],
        
        "source_file":
            row["source_file"],
        
        "signal_id":
            row["signal_id"],
        
        "class":
            row["class"],
        
        "window_id":
            row["window_id"],
        
        "true_label":
            true_label,
        
        "fp32_label":
            fp32_pred,
        
        "fp32_class":
            LABEL_TO_CLASS[fp32_pred],
        
        "fp32_confidence":
            fp32_conf,
        
        "int8_label":
            int8_pred,
        
        "int8_class":
            LABEL_TO_CLASS[int8_pred],
        
        "int8_confidence":
            int8_conf
    })

comparison_df = pd.DataFrame(results)

print("Comparison rows:",
      len(comparison_df))

display(comparison_df.head())

Comparison rows: 136


,recording_id,source_file,signal_id,class,window_id,true_label,fp32_label,fp32_class,fp32_confidence,int8_label,int8_class,int8_confidence
0,B007_3_X121,B007_3.mat,X121,Ball,0,1,3,Outer Race,0.991532,3,Outer Race,0.472768
1,B007_3_X121,B007_3.mat,X121,Ball,1,1,3,Outer Race,0.991519,3,Outer Race,0.472768
2,B007_3_X121,B007_3.mat,X121,Ball,2,1,3,Outer Race,0.991489,3,Outer Race,0.472768
3,B007_3_X121,B007_3.mat,X121,Ball,3,1,3,Outer Race,0.991643,3,Outer Race,0.472768
4,B007_3_X121,B007_3.mat,X121,Ball,4,1,3,Outer Race,0.991601,3,Outer Race,0.472768


In [19]:
y_true = comparison_df["true_label"]

y_fp32 = comparison_df["fp32_label"]

fp32_accuracy = accuracy_score(
    y_true,
    y_fp32
)

fp32_precision = precision_score(
    y_true,
    y_fp32,
    average="weighted",
    zero_division=0
)

fp32_recall = recall_score(
    y_true,
    y_fp32,
    average="weighted",
    zero_division=0
)

fp32_f1 = f1_score(
    y_true,
    y_fp32,
    average="weighted",
    zero_division=0
)

print("FP32 TEST PERFORMANCE")
print("=" * 45)

print(f"Accuracy : {fp32_accuracy:.4f}")
print(f"Precision: {fp32_precision:.4f}")
print(f"Recall   : {fp32_recall:.4f}")
print(f"F1-score : {fp32_f1:.4f}")

FP32 TEST PERFORMANCE
Accuracy : 0.1397
Precision: 0.0227
Recall   : 0.1397
F1-score : 0.0390


In [20]:
y_int8 = comparison_df["int8_label"]

int8_accuracy = accuracy_score(
    y_true,
    y_int8
)

int8_precision = precision_score(
    y_true,
    y_int8,
    average="weighted",
    zero_division=0
)

int8_recall = recall_score(
    y_true,
    y_int8,
    average="weighted",
    zero_division=0
)

int8_f1 = f1_score(
    y_true,
    y_int8,
    average="weighted",
    zero_division=0
)

print("INT8 TEST PERFORMANCE")
print("=" * 45)

print(f"Accuracy : {int8_accuracy:.4f}")
print(f"Precision: {int8_precision:.4f}")
print(f"Recall   : {int8_recall:.4f}")
print(f"F1-score : {int8_f1:.4f}")

INT8 TEST PERFORMANCE
Accuracy : 0.1397
Precision: 0.0227
Recall   : 0.1397
F1-score : 0.0390


In [21]:
accuracy_difference = (
    fp32_accuracy -
    int8_accuracy
)

print("FP32 accuracy:",
      fp32_accuracy)

print("INT8 accuracy:",
      int8_accuracy)

print(
    "Accuracy difference:",
    accuracy_difference
)

FP32 accuracy: 0.13970588235294118
INT8 accuracy: 0.13970588235294118
Accuracy difference: 0.0


In [22]:
comparison_df["prediction_match"] = (
    comparison_df["fp32_label"] ==
    comparison_df["int8_label"]
)

agreement_count = (
    comparison_df["prediction_match"]
    .sum()
)

total_predictions = len(
    comparison_df
)

agreement_percentage = (
    agreement_count /
    total_predictions *
    100
)

print("FP32/INT8 Prediction Agreement")
print("=" * 45)

print(
    "Matching predictions:",
    agreement_count
)

print(
    "Total predictions:",
    total_predictions
)

print(
    f"Agreement: {agreement_percentage:.2f}%"
)

FP32/INT8 Prediction Agreement
Matching predictions: 136
Total predictions: 136
Agreement: 100.00%


In [23]:
disagreements = comparison_df[
    ~comparison_df["prediction_match"]
]

print(
    "Number of disagreements:",
    len(disagreements)
)

if len(disagreements) > 0:
    display(disagreements)
else:
    print(
        "No FP32/INT8 prediction disagreements."
    )

Number of disagreements: 0
No FP32/INT8 prediction disagreements.


In [24]:
cm_int8 = confusion_matrix(
    y_true,
    y_int8,
    labels=[0, 1, 2, 3]
)

print("INT8 CONFUSION MATRIX")
print("=" * 45)

print(cm_int8)

INT8 CONFUSION MATRIX
[[ 0  0  0 79]
 [ 0  0  0 19]
 [ 0 19  0  0]
 [ 0  0  0 19]]


In [25]:
print(
    classification_report(
        y_true,
        y_int8,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        zero_division=0
    )
)

              precision    recall  f1-score   support

     Healthy       0.00      0.00      0.00        79
        Ball       0.00      0.00      0.00        19
  Inner Race       0.00      0.00      0.00        19
  Outer Race       0.16      1.00      0.28        19

    accuracy                           0.14       136
   macro avg       0.04      0.25      0.07       136
weighted avg       0.02      0.14      0.04       136



In [26]:
INT8_RESULTS_DIR = (
    RESULTS_DIR / "quantized"
)

INT8_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

comparison_path = (
    INT8_RESULTS_DIR /
    "fp32_vs_int8_test_predictions.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print("Saved:")
print(comparison_path)

Saved:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized\fp32_vs_int8_test_predictions.csv


In [27]:
metric_comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    
    "FP32": [
        fp32_accuracy,
        fp32_precision,
        fp32_recall,
        fp32_f1
    ],
    
    "INT8": [
        int8_accuracy,
        int8_precision,
        int8_recall,
        int8_f1
    ]
})

metric_comparison["Difference"] = (
    metric_comparison["FP32"] -
    metric_comparison["INT8"]
)

display(metric_comparison)

,Metric,FP32,INT8,Difference
0,Accuracy,0.139706,0.139706,0.0
1,Precision,0.022687,0.022687,0.0
2,Recall,0.139706,0.139706,0.0
3,F1-score,0.039035,0.039035,0.0


In [28]:
metrics_path = (
    INT8_RESULTS_DIR /
    "fp32_vs_int8_metrics.csv"
)

metric_comparison.to_csv(
    metrics_path,
    index=False
)

print("Saved:")
print(metrics_path)

Saved:
c:\Users\armaa\OneDrive\Desktop\TinyML-Vibration-Anomaly-Detection\results\quantized\fp32_vs_int8_metrics.csv


In [29]:
print("=" * 65)
print("STEP 225 — INT8 ACCURACY VERIFICATION")
print("=" * 65)

print("\nTEST WINDOWS:", len(comparison_df))

print("\nFP32:")
print(f"Accuracy : {fp32_accuracy:.4f}")
print(f"Precision: {fp32_precision:.4f}")
print(f"Recall   : {fp32_recall:.4f}")
print(f"F1-score : {fp32_f1:.4f}")

print("\nINT8:")
print(f"Accuracy : {int8_accuracy:.4f}")
print(f"Precision: {int8_precision:.4f}")
print(f"Recall   : {int8_recall:.4f}")
print(f"F1-score : {int8_f1:.4f}")

print(
    "\nAccuracy difference:",
    f"{accuracy_difference:.4f}"
)

print(
    "Prediction agreement:",
    f"{agreement_percentage:.2f}%"
)

print(
    "Prediction disagreements:",
    len(disagreements)
)

print("\nStep 225 completed.")

STEP 225 — INT8 ACCURACY VERIFICATION

TEST WINDOWS: 136

FP32:
Accuracy : 0.1397
Precision: 0.0227
Recall   : 0.1397
F1-score : 0.0390

INT8:
Accuracy : 0.1397
Precision: 0.0227
Recall   : 0.1397
F1-score : 0.0390

Accuracy difference: 0.0000
Prediction agreement: 100.00%
Prediction disagreements: 0

Step 225 completed.
